<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.2-transient-heat/Ex08.2_04_inverse_alpha.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.2 · Notebook 04 — Recovering the Diffusivity

**Paired with L8.2 · Dynamic Heat**

$\alpha$ is invisible in a steady state and plainly visible in a cooling
curve. Recover it from synthetic sensor data.

This is the first time in the course that the network is asked to find a
**physical constant** rather than a field. The unknown joins the trainable
parameters and the loss gains a data term; nothing else changes. It returns in
L11.2 and again in Ex_12.1.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.2-transient-heat/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The truth, and the measurements of it

Five sensors at random positions, each read twelve times through the **early**
transient, with a little noise on every reading. That word "early" is the whole
exercise — hold on to it until the question at the end.

In [ ]:
C_TRUE = 0.7
xyt_f = to_tensor(pb.plate_spacetime_points(3000), requires_grad=True)

# synthetic sensors: five points, sampled through the early transient
rng = np.random.default_rng(3)
sx, sy = rng.random(5), rng.random(5)
st_ = np.linspace(0.01, 0.25, 12)
P = np.array([[x, y, t] for x, y in zip(sx, sy) for t in st_])
Td = pb.exact_transient(P[:, 0], P[:, 1], P[:, 2], C_TRUE)
Td += 0.002 * rng.standard_normal(Td.shape)          # measurement noise

xyt_d = to_tensor(P)
T_d = to_tensor(Td.reshape(-1, 1))
print("sensor readings:", T_d.shape[0])
print(f"reading range  : {Td.min():.4f} .. {Td.max():.4f}")
print(f"noise          : 0.002, i.e. {0.002 / np.abs(Td).max() * 100:.2f}% "
      f"of the largest reading")

## 2 · Make alpha trainable and add a data term

$\alpha$ is positive, so optimise $\log\alpha$ and let $\alpha = e^{\log\alpha}$.
The optimiser then cannot walk into a negative diffusivity, which is a
non-physical parameter that produces a well-posed-looking but backwards
equation.

### Your turn

In [ ]:
set_seed(88)
model = MLP(n_in=3, n_hidden=40, n_layers=4)
log_alpha = torch.zeros(1, device=DEVICE, requires_grad=True)   # alpha = exp(.)


def residual(model, xyt, alpha):
    T = model(xyt)
    # TODO: return grad(T, xyt)[:, 2:3] - alpha * (d2(T, xyt, 0) + d2(T, xyt, 1))
    raise NotImplementedError


def loss_fn():
    alpha = torch.exp(log_alpha)
    # TODO: L_pde = mse(residual(model, xyt_f, alpha))
    # TODO: L_dat = mse(model(xyt_d) - T_d)
    # TODO: return L_pde + L_dat
    raise NotImplementedError


# NOTE: log_alpha must be handed to the optimiser too. train_two_stage only
# sees model.parameters(), so you have two routes:
#
#   (a) register it on the model, and it is optimised like any weight:
#         model.log_alpha = torch.nn.Parameter(torch.zeros(1, device=DEVICE))
#       then read alpha = torch.exp(model.log_alpha) inside the loss and call
#       train_two_stage as usual;
#
#   (b) write your own short loop for this notebook:
#         opt = torch.optim.Adam(list(model.parameters()) + [log_alpha], lr=1e-3)
#
# Record the loss history as `history` (a list of floats is fine) and the
# recovered value as `alpha_hat`, a float -- the cells below use both.

## 3 · What you recovered

In [ ]:
print(f"  true alpha       : {C_TRUE:.4f}")
print(f"  recovered alpha  : {alpha_hat:.4f}")
print(f"  relative error   : {abs(alpha_hat - C_TRUE) / C_TRUE * 100:.2f}%")

fig, ax = plt.subplots(1, 2, figsize=(12.0, 3.6))
ax[0].semilogy(np.asarray(history), lw=1.4, color=CYCLE[0])
ax[0].set_xlabel("step"); ax[0].set_ylabel("loss"); ax[0].set_title("training")
ax[0].grid(alpha=0.25, which="both")

tt = np.linspace(0.0, 0.4, 200)
ax[1].plot(tt, np.exp(-2 * np.pi ** 2 * C_TRUE * tt), lw=1.8, color="#111111",
           label=f"true, c = {C_TRUE}")
ax[1].plot(tt, np.exp(-2 * np.pi ** 2 * alpha_hat * tt), lw=1.8, ls="--",
           color=CYCLE[0], label=f"recovered, c = {alpha_hat:.4f}")
ax[1].axvspan(st_[0], st_[-1], color="#f4a300", alpha=0.15,
              label="sensor window")
ax[1].set_xlabel("t"); ax[1].set_ylabel("amplitude")
ax[1].set_title("the cooling curve the data saw")
ax[1].legend(frameon=False, fontsize=9); ax[1].grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 4 · Save

In [ ]:
os.makedirs("Ex08.2_outputs", exist_ok=True)
path = os.path.join("Ex08.2_outputs", "nb04_inverse.npz")
np.savez(path, alpha_true=C_TRUE, alpha_hat=float(alpha_hat),
         sensor_t=st_, loss=np.asarray(history, dtype=float))
print("wrote", path)

## 5 · The question that matters

**Question.** Move the sensor times to `np.linspace(0.6, 1.0, 12)` — the late,
settled part of the transient — and retrain. What happens to the recovered
$\alpha$, and why? *(slide 20)*

Do it. Report the number you get, not the number you expect. The amplitude
table in notebook 00 tells you what the sensors are reading in that window, and
the noise you added is 0.002.

This is **identifiability**: a parameter that the data does not constrain will
still be reported by the optimiser, with no warning attached. It recurs in
every later inverse problem in the course.

Next: **notebook 05**, the report.